# Review 1. Preprocessing at several smoothing scales

The notebook 1 pipeline run at 10, 20, 30, 40 and 50 A Gaussian smoothing, so
that the hyperparameter review can ask how much the UMAP embedding depends on
that choice.  Notebook 1 is left untouched; these tables go to `data/review/`.

| input | what it provides |
| --- | --- |
| `metadata/params.csv` | one row per spectrum: the table built in notebook 0 |
| `spectrum/<filename_csv>` | the raw spectra |
| `data/F30.csv`, `data/F30_norm.csv` | the notebook 1 tables, for the check at the end |

| output | how it is built |
| --- | --- |
| `data/review/F<S>.csv` | rest frame -> Gaussian smooth (S A) -> cubic interpolation |
| `data/review/F<S>_norm.csv` | rest frame -> median normalization -> LOWESS continuum division -> Gaussian smooth (S A) -> cubic interpolation |

The 30 A pair reproduces the notebook 1 tables.


## Imports

In [1]:
import os

import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
from statsmodels.nonparametric.smoothers_lowess import lowess
from tqdm import tqdm

In [2]:
import astropy.units as u
from specutils import Spectrum1D
from specutils.manipulation import gaussian_smooth

import warnings
from astropy.utils.exceptions import AstropyDeprecationWarning

warnings.filterwarnings("ignore", category=AstropyDeprecationWarning)

## Configuration


In [3]:
# Input / output locations
PARAMS_PATH = './metadata/params.csv'
SPECTRUM_DIR = './spectrum'

SPECTRUM_TABLE_DIR = './data/review'

# The notebook 1 tables the 30 A pair built here is checked against
REFERENCE_TABLE_DIR = './data'

# Common rest-frame wavelength grid [Angstrom]
GRID_MIN_AA = 5000
GRID_MAX_AA = 7000
GRID_N_POINTS = 1000

# Legacy peak_reduction.ipynb parameters
INTERPOLATION_KIND = 'cubic'
LOWESS_FRACTION = 0.3

# Gaussian smoothing scales [Angstrom].  Notebook 1 uses 30 only; each scale here
# gives a pair of tables named after it, F<scale>.csv and F<scale>_norm.csv.
SMOOTHING_SCALES_AA = [20, 30, 40]

# Column label format: '<SN name>%%%<SN type>$$$<phase>'
NAME_TYPE_SEPARATOR = '%%%'
TYPE_PHASE_SEPARATOR = '$$$'

FLUX_UNIT = u.erg / u.cm ** 2 / u.s / u.AA

WAVELENGTH_GRID_AA = np.linspace(GRID_MIN_AA, GRID_MAX_AA, GRID_N_POINTS)

## Preprocessing steps

The notebook 1 functions, with the smoothing scale passed down as
`smoothing_scale_AA` instead of hard-coded.  The legacy order is kept: the
continuum is removed on the native rest-frame sampling, **before** smoothing and
before interpolation onto the common grid, so `F<S>_norm` is not made from
`F<S>`.


In [4]:
def make_column_label(sn_name, sn_type, phase):
    """Encode the identity of a supernova into a single column label."""
    return f'{sn_name}{NAME_TYPE_SEPARATOR}{sn_type}{TYPE_PHASE_SEPARATOR}{phase}'

In [5]:
def load_rest_frame_arrays(spectrum_path, redshift):
    """Replicate the input/rest-frame part of legacy `read_spec()`."""
    # Legacy code reads the full CSV, then uses columns 0 and 1.
    data = pd.read_csv(spectrum_path, header=None)
    data.drop_duplicates(subset=0, inplace=True)

    AA = data[0] / (1.0 + redshift)
    flux = data[1] * (1.0 + redshift)

    return AA, flux


In [6]:
def normalization(AA, flux, Del_Lambda, smoothing_scale_AA):
    """Replicate legacy `peak_reduction.ipynb::normalization()`.

    Important: `bin_ispec` is intentionally computed but not used afterwards,
    exactly as in the legacy notebook.
    """
    f = interp1d(AA, flux, fill_value='extrapolate')
    ispec = f(AA)
    ispec = ispec / np.median(ispec)

    ispec_astropy = Spectrum1D(
        flux=ispec << FLUX_UNIT,
        spectral_axis=AA.to_numpy() << u.AA,
    )

    # Intentionally unused: retained to match the old function-call sequence.
    bin_ispec = gaussian_smooth(
        ispec_astropy,
        stddev=int(smoothing_scale_AA / Del_Lambda + 1),
    )

    ispec_lowess = ispec_astropy.flux.value
    vals = lowess(ispec_lowess, AA, frac=0.3)
    smoothed_ispec = vals[:, 1]

    spe = ispec / smoothed_ispec
    spe[~np.isfinite(spe)] = 0

    spec = Spectrum1D(
        flux=spe << FLUX_UNIT,
        spectral_axis=AA.to_numpy() << u.AA,
    )

    return spec

In [7]:
def make_table_on_common_grid(AA, flux, Del_Lambda, interpol_AA, smoothing_scale_AA,
                              continuum_remove=False):
    """Replicate the legacy `read_spec()` operations relevant to F30, at any scale.

    continuum_remove=False:
        rest-frame spectrum -> Gaussian smoothing -> cubic interpolation

    continuum_remove=True:
        rest-frame spectrum -> legacy normalization()/LOWESS division
        -> Gaussian smoothing -> cubic interpolation
    """
    if continuum_remove:
        spec = normalization(AA, flux, Del_Lambda, smoothing_scale_AA)
    else:
        spec = Spectrum1D(
            flux=flux.to_numpy() << FLUX_UNIT,
            spectral_axis=AA.to_numpy() << u.AA,
        )

    smoothed = gaussian_smooth(
        spec,
        stddev=int(smoothing_scale_AA / Del_Lambda + 1),
    )

    interpol_smoothed = interp1d(
        smoothed.spectral_axis,
        smoothed.flux,
        kind='cubic',
    )

    return interpol_smoothed(interpol_AA)

In [8]:
def preprocess_spectrum(spectrum_path, redshift, Del_Lambda,
                        interpol_AA, smoothing_scale_AA, continuum_remove=False):
    """Read one raw spectrum and produce its legacy-compatible smoothed array."""
    AA, flux = load_rest_frame_arrays(spectrum_path, redshift)
    return make_table_on_common_grid(
        AA,
        flux,
        Del_Lambda,
        interpol_AA,
        smoothing_scale_AA,
        continuum_remove=continuum_remove,
    )

In [9]:
def build_spectrum_table(metadata, wavelength_grid_AA, smoothing_scale_AA, continuum_remove,
                         spectrum_dir=SPECTRUM_DIR):
    """Build either the legacy F<S> or the legacy F<S>_norm table directly from raw spectra."""
    columns = {'AA': wavelength_grid_AA}

    for supernova in tqdm(metadata.itertuples(index=False), total=len(metadata)):
        spectrum_path = os.path.join(spectrum_dir, supernova.filename_csv)

        flux = preprocess_spectrum(
            spectrum_path,
            supernova.redshift,
            supernova.Del_Lambda,
            wavelength_grid_AA,
            smoothing_scale_AA,
            continuum_remove=continuum_remove,
        )

        label = make_column_label(
            supernova.SN_name,
            supernova.SN_type,
            supernova.phase,
        )
        columns[label] = flux

    return pd.DataFrame(columns)

## Build the data sets


In [10]:
def spectrum_table_name(smoothing_scale_AA):
    """`F30.csv` for 30 A: the notebook 1 name, with the scale in it."""
    return f'F{smoothing_scale_AA}.csv'


def normalized_table_name(smoothing_scale_AA):
    """`F30_norm.csv` for 30 A: the notebook 1 name, with the scale in it."""
    return f'F{smoothing_scale_AA}_norm.csv'


def build_smoothed_table(metadata, smoothing_scale_AA, wavelength_grid_AA=WAVELENGTH_GRID_AA):
    """Legacy `../data/spec/F30.csv` equivalent, at the scale given."""
    return build_spectrum_table(
        metadata,
        wavelength_grid_AA,
        smoothing_scale_AA,
        continuum_remove=False,
    )

In [11]:
def build_normalized_table(metadata, smoothing_scale_AA, wavelength_grid_AA=WAVELENGTH_GRID_AA):
    """Legacy `../data/norm/F30.csv` equivalent, at the scale given, saved as `F<S>_norm.csv`."""
    return build_spectrum_table(
        metadata,
        wavelength_grid_AA,
        smoothing_scale_AA,
        continuum_remove=True,
    )

In [12]:
def save_table(table, output_dir, filename):
    """Write a table to `output_dir/filename`, creating the directory if needed."""
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, filename)
    table.to_csv(output_path, index=False)
    return output_path

In [13]:
metadata = pd.read_csv(PARAMS_PATH)
print(f'{len(metadata)} spectra listed in {PARAMS_PATH}')
metadata[['SN_name', 'SN_type', 'phase', 'redshift', 'Del_Lambda', 'filename_csv']].head()

119 spectra listed in ./metadata/params.csv


,SN_name,SN_type,phase,redshift,Del_Lambda,filename_csv
0,SN1994D,NV,0.35,0.001494,1.467083,SN1994D_1994-03-21_08-24-00_FLWO-1.5m_FAST_CfA...
1,SN1994S,NV,1.00,0.015177,1.969612,SN_1994S_1994-06-16_00-00-00_Lick-3m_KAST_UCB-...
2,SN1996ai,NV,-0.10,0.002900,1.992704,SN_1996ai_1996-06-21_00-00-00_Lick-3m_KAST_UCB...
3,SN1996C,NV,1.98,0.027000,1.430809,1996C_1996-02-17_11-31-12_FLWO-1.5m_FAST_CfA-I...
4,SN1996X,NV,0.31,0.008876,1.456511,SN1996X_1996-04-18_07-26-24_FLWO-1.5m_FAST_CfA...


In [14]:
for smoothing_scale_AA in SMOOTHING_SCALES_AA:
    print(f'--- {smoothing_scale_AA} A smoothing')

    spectrum_table = build_smoothed_table(metadata, smoothing_scale_AA)
    output_path = save_table(spectrum_table, SPECTRUM_TABLE_DIR,
                             spectrum_table_name(smoothing_scale_AA))
    print(
        f'wrote {spectrum_table.shape[0]} wavelengths x '
        f'{spectrum_table.shape[1] - 1} spectra to {output_path}'
    )

    # IMPORTANT: build the _norm table again from the RAW spectra.
    # Do not continuum-normalize `spectrum_table`, because that would reverse the
    # legacy operation order.
    normalized_table = build_normalized_table(metadata, smoothing_scale_AA)
    normalized_output_path = save_table(normalized_table, SPECTRUM_TABLE_DIR,
                                        normalized_table_name(smoothing_scale_AA))
    print(f'wrote the legacy-compatible continuum-removed spectra to {normalized_output_path}')

--- 20 A smoothing


100%|██████████| 119/119 [00:00<00:00, 142.47it/s]


wrote 1000 wavelengths x 119 spectra to ./data/review/F20.csv


100%|██████████| 119/119 [00:48<00:00,  2.48it/s]


wrote the legacy-compatible continuum-removed spectra to ./data/review/F20_norm.csv
--- 30 A smoothing


...

wrote the legacy-compatible continuum-removed spectra to ./data/review/F30_norm.csv
--- 40 A smoothing


100%|██████████| 119/119 [00:00<00:00, 161.50it/s]


wrote 1000 wavelengths x 119 spectra to ./data/review/F40.csv


100%|██████████| 119/119 [00:47<00:00,  2.49it/s]


wrote the legacy-compatible continuum-removed spectra to ./data/review/F40_norm.csv


### Check

The 30 A pair built here should reproduce the notebook 1 tables exactly.


In [15]:
for review_name in [spectrum_table_name(30), normalized_table_name(30)]:
    reference_path = os.path.join(REFERENCE_TABLE_DIR, review_name)
    review = pd.read_csv(os.path.join(SPECTRUM_TABLE_DIR, review_name))
    reference = pd.read_csv(reference_path)
    largest_difference = (review - reference).abs().max().max()
    print(f'{review_name} vs {reference_path}: largest difference {largest_difference:.3g}')

F30.csv vs ./data/F30.csv: largest difference 0
F30_norm.csv vs ./data/F30_norm.csv: largest difference 0
